# ANA Educa??o Financeira ? Base Sumarizada por Cliente ? V2

Fluxo consolidado para gerar a base oficial `df_base_sumarizada_cliente`, com uma linha por cliente na janela mensal.

Esta vers?o V2 preserva a estrutura da base e altera as regras de pontua??o de or?amento, perfil e pontua??o final por tema. A concentra??o segue como principal sinal do per?odo; o or?amento refor?a o contexto financeiro; e o perfil atua apenas como personaliza??o, sem criar sozinho um tema relevante.

A base final re?ne:
- resumo t?cnico das transa??es;
- perfil de renda e perfil financeiro;
- blocos de entrada e sa?da por classifica??o;
- resultado do or?amento e seu status;
- refer?ncias, pontua??es intermedi?rias e pontua??es finais por tema;
- marca??o de movimenta??o agro e flag de participa??o no radar.

Ap?s a consolida??o, os indicadores derivados usam `vw_base_sumarizada_cliente`. A marca??o agro ? obtida da classifica??o transacional j? estruturada no mapa de categorias.


## Escopo e decisões consolidadas

### Estrutura do fluxo

```text
vw_base_transacoes
        ↓
vw_resumo_transacoes_cliente          → resumo técnico por natureza C/D
vw_blocos_classificacao_cliente       → blocos por CD_CLASSIFICACAO_CATEGORIA
        ↓
vw_base_sumarizada_cliente            → base oficial por cliente
        ↓
vw_resultado_orcamento                → resultado derivado da própria sumarizada
        ↓
vw_base_sumarizada_cliente            → referências e pontuações por tema
        ↓
FL_TEM_MOV_AGRO + FL_PARTICIPA_RADAR  → marcação e elegibilidade técnica
        ↓
ana_edu_fin_cli                       → tabela final
```

### Regras mantidas

- O resumo técnico usa `CD_NTZ_CTB_TRAN` para quantidades e valores gerais de crédito e débito.
- Os blocos classificados usam exclusivamente `CD_CLASSIFICACAO_CATEGORIA`:
  - `0` a `4`: entradas;
  - `5` a `9`: saídas.
- O resultado do orçamento usa `VL_ENT_TOTAL` e `VL_SAI_TOTAL`.
- Todas as divisões entre valores usam `NULLIF` e `COALESCE`; sem denominador válido, o resultado é `0`.
- `FL_AGRO_CATEGORIA` é mantida apenas na camada transacional e marca as categorias agro no dicionário.
- `FL_TEM_MOV_AGRO` recebe `S` quando houver movimentação `C` ou `D` em categoria agro.
- Cliente com `FL_TEM_MOV_AGRO = 'S'` não participa do radar.
- A estrutura final tem **71 colunas**, em uma linha por `CD_CLI`.


## 1. Conexão

In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()

    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="etl-vinculacao-mf-insights",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": ambiente,
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            "spark.serializer":
                "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "240",
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive":"true",
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
try:
    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%%spark

import os

ambiente = ler_variavel_ambiente_spark("AMBIENTE").upper()
dominio = ler_variavel_ambiente_spark("DOMINIO").lower()
hoje = ler_variavel_ambiente_spark("HOJE")

if ambiente == "MODELAGEM":
    sandbox = ler_variavel_ambiente_spark("SANDBOX").lower()
    database = f"sbx_{sandbox}"
else:
    database = f"hive_{dominio}"

env_spark = dict(os.environ)

cliente_db2 = criar_cliente_db2_spark(env=env_spark)


## 2. Configuração e janela de processamento

In [ ]:
%%spark

from datetime import datetime
from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.window import Window

nome_tabela_saida = "ana_edu_fin_cli"
nome_tabela = f"{database}.{nome_tabela_saida}"

fetchsize_db2 = 10_000
num_partitions_db2 = 16
num_partitions_transacoes = 20
partitions_hive = 10
partitions_stage_default = 20

materializar_etapas = True
limpar_cache_intermediario = True
limpar_cache_publicacao = True
storage_level_materializacao = StorageLevel.MEMORY_AND_DISK

diagnostico_opt_in = False

# Faixa padrão para colunas de identificação de cliente.
cd_cli_min = 1
cd_cli_max = 999_999_999

# Faixas do resultado do orçamento:
# - 0,25: distância a partir da qual o superávit/déficit é forte.
# - 0,05: faixa de neutralidade em torno do equilíbrio.
faixa_resultado_forte = 0.25
faixa_resultado_neutro = 0.05

pc_limite_superavit_forte = 1 - faixa_resultado_forte
pc_limite_neutro_inferior = 1 - faixa_resultado_neutro
pc_limite_neutro_superior = 1 + faixa_resultado_neutro
pc_limite_deficit_forte = 1 + faixa_resultado_forte

tabela_spark = nome_tabela

pc_ref_gen = 0.750000
pc_ref_ess = 0.500000
pc_ref_flex = 0.300000
pc_ref_res = 0.200000
pc_ref_cred = 0.300000


In [ ]:
%%spark

df_datas = (
    spark.createDataFrame([(1,)], ["id"])
    .withColumn(
        "data_atual",
        F.to_date(F.lit(hoje), "yyyyMMdd"),
    )
    .withColumn(
        "data_fim",
        F.date_sub(F.col("data_atual"), 1),
    )
    .withColumn(
        "data_ini",
        F.date_sub(F.col("data_fim"), 29),
    )
    .withColumn(
        "data_recorte",
        F.date_sub(F.col("data_ini"), 1),
    )
    .select(
        "data_atual",
        "data_ini",
        "data_fim",
        "data_recorte",
    )
)

linha_datas = df_datas.collect()[0]

data_atual = linha_datas["data_atual"].strftime("%Y-%m-%d")
data_ini = linha_datas["data_ini"].strftime("%Y-%m-%d")
data_fim = linha_datas["data_fim"].strftime("%Y-%m-%d")
data_recorte = linha_datas["data_recorte"].strftime("%Y-%m-%d")

ano_etl = linha_datas["data_fim"].year

print(f"Data atual: {data_atual}")
print(f"Data de início da query: {data_ini}")
print(f"Data final da query: {data_fim}")
print(f"Data de recorte da inclusão: {data_recorte}")
print(f"Ano do ETL: {ano_etl}")

## 3. Base transacional

In [ ]:
%%spark

query_base_transacoes = f"""
SELECT
    a.CD_CLI,

    a.CD_PRD,

    CASE
        WHEN a.CD_PRD = 6 THEN 'CONTA CORRENTE'
        WHEN a.CD_PRD = 9 THEN 'CARTAO'
        ELSE NULL
    END AS NM_PRD,

    a.VL_TRAN,

    a.CD_TIP_MOE_CRR,

    CASE
        WHEN a.CD_TIP_MOE_CRR IS NULL THEN CAST(NULL AS VARCHAR(100))
        ELSE CAST(NULL AS VARCHAR(100))
    END AS NM_TIP_MOE_CRR,

    a.CD_NTZ_CTB_TRAN,

    CASE
        WHEN a.CD_NTZ_CTB_TRAN = 'C' THEN 'CREDITO'
        WHEN a.CD_NTZ_CTB_TRAN = 'D' THEN 'DEBITO'
        ELSE NULL
    END AS NM_TP_LANCAMENTO,

    a.CD_CTGR_TRAN_OGNL

FROM DB2GFP.TRAN_RLZD_INST_PCT a

WHERE a.DT_TRAN BETWEEN '{data_ini}' AND '{data_fim}'

  -- Mantém transações incluídas a partir do dia anterior
  -- ao início do período financeiro analisado.
  AND DATE(a.TS_INCL_TRAN) >= '{data_recorte}'

  -- Mantém somente transações no estado operacional esperado.
  AND a.CD_EST_TRAN_INST = 0

  -- Recorte legado temporário para estudo.
  AND a.NR_PTC BETWEEN 80 AND 85
"""

df_base_transacoes = cliente_db2.run_select(
    query_base_transacoes,
    fetchsize=fetchsize_db2,
    partition_column="CD_CLI",
    lower_bound=1,
    upper_bound=999999999,
    num_partitions=num_partitions_transacoes,
)

## 4. Classificação transacional por categoria e natureza

In [ ]:
%%spark

# ============================================================
# 1. Dicionários de códigos por tipo de classificação
# ============================================================

codigos_classificacao_credito = {
    "Transferência / entrada indefinida": 0,
    "Receita / rendimento / benefício": 1,
    "Restituição / estorno / reembolso / ajuste": 2,
    "Resgate de investimento": 3,
    "Crédito tomado / liberação de crédito": 4,
}

codigos_classificacao_debito = {
    "Neutro / não classificado": 5,
    "Essenciais": 6,
    "Flexíveis": 7,
    "Futuro": 8,
    "Dívidas / crédito / custo financeiro": 9,
}


# ============================================================
# 2. Mapa de categorias transacionais
#
# Estrutura:
# (
#     CD_GRUPO_CATEGORIA,
#     CD_CTGR_TRAN_OGNL,
#     NM_CATEGORIA,
#     NM_CLASSIFICACAO_CREDITO,
#     NM_CLASSIFICACAO_DEBITO,
#     FL_AGRO_CATEGORIA
# )
# ============================================================

base_variaveis_classificacao = [
    (0, 0, "Sem categoria", "Transferência / entrada indefinida", "Neutro / não classificado", "N"),
    (0, 83, "Sem categoria", "Transferência / entrada indefinida", "Neutro / não classificado", "N"),

    (1, 1, "Salário", "Receita / rendimento / benefício", "Neutro / não classificado", "N"),
    (1, 2, "Vale Alimentação", "Receita / rendimento / benefício", "Neutro / não classificado", "N"),
    (1, 3, "Restituição de IR", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado", "N"),
    (1, 4, "Bonificação", "Receita / rendimento / benefício", "Neutro / não classificado", "N"),
    (1, 5, "Outros Rendimentos", "Receita / rendimento / benefício", "Neutro / não classificado", "N"),

    (2, 6, "Água", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (2, 7, "Eletricidade e Gás", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (2, 9, "Compra de Imóvel", "Crédito tomado / liberação de crédito", "Dívidas / crédito / custo financeiro", "N"),
    (2, 10, "Aluguel e Condomínio", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (2, 11, "Móveis e Utensílios", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (2, 12, "Serviços e Manutenção", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (2, 13, "Empregados", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (2, 14, "Animais e Pets", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (2, 3790, "Seguro Residencial", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),

    (3, 15, "Educação Superior", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (3, 16, "Colégio", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (3, 17, "Idiomas", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (3, 18, "Publicações e Papelaria", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (3, 20, "Outros Gastos", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),

    (4, 21, "Viagens e Lazer", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (4, 22, "Esportes e Academia", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (4, 25, "Cultura e Entretenimento", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (4, 26, "Publicações Digitais", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (4, 61, "Jogos e Loterias", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),

    (5, 27, "Plano de Saúde", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (5, 28, "Serviços de Saúde", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (5, 29, "Dentista", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (5, 30, "Farmácias e Drogarias", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),

    (6, 32, "Feira e Supermercado", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (6, 35, "Bar", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),

    (7, 36, "Compra de Veículo", "Crédito tomado / liberação de crédito", "Dívidas / crédito / custo financeiro", "N"),
    (7, 37, "Combustível", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (7, 38, "Estacionamento e Pedágio", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (7, 39, "Seguro de Veículo", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (7, 40, "Serviços e Manutenção", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (7, 41, "Transporte Urbano e Apps", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),

    (8, 42, "Vestuário e Acessórios", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (8, 43, "Cuidado Pessoal e Beleza", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (8, 44, "Compras Diversas", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado", "N"),
    (8, 45, "Pensão Alimentícia", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (8, 46, "Seguros e Previdência", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado", "N"),
    (8, 47, "Doação", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (8, 48, "Gasto com Familiares", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (8, 49, "Presentes", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (8, 60, "Serviços diversos", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),
    (8, 4417, "Empréstimos e Prestações", "Crédito tomado / liberação de crédito", "Dívidas / crédito / custo financeiro", "N"),

    (9, 51, "Telefonia e Internet", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (9, 53, "Assinatura TV e Streaming", "Restituição / estorno / reembolso / ajuste", "Flexíveis", "N"),

    (10, 54, "IPTU", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (10, 55, "IPVA e Gastos Detran", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (10, 56, "Imposto de Renda", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (10, 57, "ISS(Imposto sobre Serviços)", "Restituição / estorno / reembolso / ajuste", "Essenciais", "N"),
    (10, 58, "GPS(Guia de Previdência Social)", "Restituição / estorno / reembolso / ajuste", "Futuro", "N"),
    (10, 59, "Serviços Financeiros", "Restituição / estorno / reembolso / ajuste", "Dívidas / crédito / custo financeiro", "N"),
    (10, 3787, "IOF", "Restituição / estorno / reembolso / ajuste", "Dívidas / crédito / custo financeiro", "N"),
    (10, 3788, "Encargos e Tarifas", "Restituição / estorno / reembolso / ajuste", "Dívidas / crédito / custo financeiro", "N"),

    (11, 279, "Gastos Diversos", "Transferência / entrada indefinida", "Neutro / não classificado", "N"),
    (11, 39434, "Cheque", "Transferência / entrada indefinida", "Neutro / não classificado", "N"),
    (11, 39435, "Saque", "Transferência / entrada indefinida", "Neutro / não classificado", "N"),
    (11, 39436, "Transferência", "Transferência / entrada indefinida", "Neutro / não classificado", "N"),
    (11, 39437, "Boletos Diversos", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado", "N"),

    (12, 111, "Cartão de Crédito", "Restituição / estorno / reembolso / ajuste", "Dívidas / crédito / custo financeiro", "N"),

    (13, 448977, "Aplicação", "Restituição / estorno / reembolso / ajuste", "Futuro", "N"),
    (13, 448978, "Resgate de Investimentos", "Resgate de investimento", "Neutro / não classificado", "N"),

    (14, 300, "Receitas Agro", "Receita / rendimento / benefício", "Neutro / não classificado", "S"),
    (14, 310, "Criações", "Receita / rendimento / benefício", "Neutro / não classificado", "S"),
    (14, 330, "Cultivos", "Receita / rendimento / benefício", "Neutro / não classificado", "S"),
    (14, 350, "Insumos", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado", "S"),
    (14, 370, "Apoio Produtivo", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado", "S"),
]


# ============================================================
# 3. Dicionário final de classificação
#
# Chave:
# (CD_CTGR_TRAN_OGNL, CD_NTZ_CTB_TRAN)
#
# Valor:
# (
#     CD_CLASSIFICACAO_CATEGORIA,
#     NM_CLASSIFICACAO_CATEGORIA,
#     FL_AGRO_CATEGORIA
# )
# ============================================================

mapa_classificacao_categoria = {}

for (
    _,
    cd_ctgr_tran_ognl,
    _,
    nm_classificacao_credito,
    nm_classificacao_debito,
    fl_agro_categoria,
) in base_variaveis_classificacao:

    cd_ctgr_tran_ognl = int(cd_ctgr_tran_ognl)

    mapa_classificacao_categoria[
        (cd_ctgr_tran_ognl, "C")
    ] = (
        int(
            codigos_classificacao_credito[
                nm_classificacao_credito
            ]
        ),
        nm_classificacao_credito,
        fl_agro_categoria,
    )

    mapa_classificacao_categoria[
        (cd_ctgr_tran_ognl, "D")
    ] = (
        int(
            codigos_classificacao_debito[
                nm_classificacao_debito
            ]
        ),
        nm_classificacao_debito,
        fl_agro_categoria,
    )


# ============================================================
# 4. Conversão do dicionário para DataFrame de apoio
# ============================================================

linhas_mapa_classificacao = [
    (
        cd_ctgr_tran_ognl,
        cd_ntz_ctb_tran,
        cd_classificacao_categoria,
        nm_classificacao_categoria,
        fl_agro_categoria,
    )
    for (
        cd_ctgr_tran_ognl,
        cd_ntz_ctb_tran,
    ), (
        cd_classificacao_categoria,
        nm_classificacao_categoria,
        fl_agro_categoria,
    ) in sorted(mapa_classificacao_categoria.items())
]

df_mapa_classificacao_categoria = spark.createDataFrame(
    linhas_mapa_classificacao,
    """
        CD_CTGR_TRAN_OGNL BIGINT,
        CD_NTZ_CTB_TRAN STRING,
        CD_CLASSIFICACAO_CATEGORIA BIGINT,
        NM_CLASSIFICACAO_CATEGORIA STRING,
        FL_AGRO_CATEGORIA STRING
    """,
)

df_mapa_classificacao_categoria.createOrReplaceTempView(
    "vw_mapa_classificacao_categoria"
)


# ============================================================
# 5. Aplicação da classificação na base transacional
# ============================================================

codigo_fallback_credito = (
    codigos_classificacao_credito[
        "Transferência / entrada indefinida"
    ]
)

codigo_fallback_debito = (
    codigos_classificacao_debito[
        "Neutro / não classificado"
    ]
)

nome_fallback_credito = (
    "Transferência / entrada indefinida"
)

nome_fallback_debito = (
    "Neutro / não classificado"
)

df_base_transacoes_classificada = (
    df_base_transacoes.alias("b")
    .join(
        F.broadcast(
            df_mapa_classificacao_categoria
        ).alias("m"),
        (
            F.col("b.CD_CTGR_TRAN_OGNL").cast("bigint")
            == F.col("m.CD_CTGR_TRAN_OGNL")
        )
        & (
            F.col("b.CD_NTZ_CTB_TRAN")
            == F.col("m.CD_NTZ_CTB_TRAN")
        ),
        "left",
    )
    .select(
        F.col("b.CD_CLI"),
        F.col("b.CD_PRD"),
        F.col("b.NM_PRD"),
        F.col("b.VL_TRAN"),
        F.col("b.CD_TIP_MOE_CRR"),
        F.col("b.NM_TIP_MOE_CRR"),
        F.col("b.CD_NTZ_CTB_TRAN"),
        F.col("b.NM_TP_LANCAMENTO"),
        F.col("b.CD_CTGR_TRAN_OGNL"),

        F.coalesce(
            F.col("m.CD_CLASSIFICACAO_CATEGORIA"),

            F.when(
                F.col("b.CD_NTZ_CTB_TRAN") == "C",
                F.lit(codigo_fallback_credito).cast("bigint"),
            )
            .when(
                F.col("b.CD_NTZ_CTB_TRAN") == "D",
                F.lit(codigo_fallback_debito).cast("bigint"),
            )
            .otherwise(
                F.lit(None).cast("bigint")
            ),
        ).alias("CD_CLASSIFICACAO_CATEGORIA"),

        F.coalesce(
            F.col("m.NM_CLASSIFICACAO_CATEGORIA"),

            F.when(
                F.col("b.CD_NTZ_CTB_TRAN") == "C",
                F.lit(nome_fallback_credito),
            )
            .when(
                F.col("b.CD_NTZ_CTB_TRAN") == "D",
                F.lit(nome_fallback_debito),
            )
            .otherwise(
                F.lit(None).cast("string")
            ),
        ).alias("NM_CLASSIFICACAO_CATEGORIA"),

        F.coalesce(
            F.col("m.FL_AGRO_CATEGORIA"),
            F.lit("N"),
        ).alias("FL_AGRO_CATEGORIA"),
    )
)


# ============================================================
# 6. Substitui a base e recria a view usada nas próximas etapas
# ============================================================

df_base_transacoes = df_base_transacoes_classificada

df_base_transacoes.createOrReplaceTempView(
    "vw_base_transacoes"
)


## 5. Resumo técnico das transações por cliente

In [ ]:
%%spark

df_resumo_transacoes_cliente = spark.sql("""
    SELECT
        CD_CLI,

        -- Quantidade total
        COUNT(*) AS QT_TRANS_TOTAL,

        -- Quantidades de entrada
        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1
            END
        ) AS QT_TRANS_ENT,

        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'C'
                 AND CD_PRD = 6
                THEN 1
            END
        ) AS QTD_TRANS_ENT_CC,

        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'C'
                 AND CD_PRD = 9
                THEN 1
            END
        ) AS QTD_TRANS_ENT_CD,

        -- Quantidades de saída
        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1
            END
        ) AS QT_TRANS_SAI,

        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'D'
                 AND CD_PRD = 6
                THEN 1
            END
        ) AS QTD_TRANS_SAI_CC,

        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'D'
                 AND CD_PRD = 9
                THEN 1
            END
        ) AS QTD_TRANS_SAI_CD,

        -- Valores de entrada
        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'C'
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_ENT,

        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'C'
                     AND CD_PRD = 6
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_ENT_CC,

        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'C'
                     AND CD_PRD = 9
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_ENT_CD,

        -- Valores de saída
        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'D'
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_SAI,

        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'D'
                     AND CD_PRD = 6
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_SAI_CC,

        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'D'
                     AND CD_PRD = 9
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_SAI_CD

    FROM vw_base_transacoes

    GROUP BY
        CD_CLI
""")

df_resumo_transacoes_cliente.createOrReplaceTempView(
    "vw_resumo_transacoes_cliente"
)


## 6. Perfil de renda e perfil financeiro

In [ ]:
%%spark

query_perfil_renda = """
SELECT
    perfil_mais_recente.CD_CLI,
    perfil_mais_recente.CD_SGM_CLI AS CD_PRFL

FROM (
    SELECT
        cs.CD_CLI,
        cs.CD_SGM_CLI,

        ROW_NUMBER() OVER (
            PARTITION BY cs.CD_CLI
            ORDER BY cs.DT_ENQ_CLI_SGM DESC
        ) AS NR_ORDEM

    FROM DB2SMI.CLI_SGM cs

    WHERE cs.CD_CRIT_SGM_CLI = 2
      AND cs.DT_ENQ_CLI_SGM > DATE('2025-01-01')
) perfil_mais_recente

WHERE perfil_mais_recente.NR_ORDEM = 1
"""

df_perfil_renda = cliente_db2.run_select(
    query_perfil_renda,
    fetchsize=fetchsize_db2,
)

df_perfil_renda.createOrReplaceTempView(
    "vw_perfil_renda"
)


In [ ]:
%%spark

df_perfil_financeiro = spark.sql("""
    SELECT
        CAST(CD_CLI AS BIGINT) AS CD_CLI,
        CAST(CD_MAC_PRFL_CLI AS BIGINT) AS CD_MAC_PRFL_CLI,
        CAST(CD_MIC_PRFL_CLI AS BIGINT) AS CD_MIC_PRFL_CLI,
        CAST(DT_REF AS DATE) AS DT_REF_PERFIL_FINANCEIRO

    FROM (
        SELECT
            CD_CLI,
            CD_MAC_PRFL_CLI,
            CD_MIC_PRFL_CLI,
            DT_REF,

            ROW_NUMBER() OVER (
                PARTITION BY CD_CLI
                ORDER BY
                    CAST(DT_REF AS DATE) DESC NULLS LAST,
                    CD_MAC_PRFL_CLI DESC,
                    CD_MIC_PRFL_CLI DESC
            ) AS NR_ORDEM

        FROM sbx_t2i2016.DVS_GRDR_FNCO_PF
    ) perfil_mais_recente

    WHERE NR_ORDEM = 1
""")

df_perfil_financeiro.createOrReplaceTempView(
    "vw_perfil_financeiro"
)


## 7. Base sumarizada oficial por cliente

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    WITH perfil_renda AS (
        SELECT
            CAST(CD_CLI AS BIGINT) AS CD_CLI,
            CAST(CD_PRFL AS INT) AS CD_PRFL,

            CASE
                WHEN CD_PRFL = 2502 THEN 'SEM PERFIL'
                WHEN CD_PRFL = 2012 THEN 'PF A'
                WHEN CD_PRFL = 2112 THEN 'PF B'
                WHEN CD_PRFL = 2602 THEN 'PF C'
                WHEN CD_PRFL = 2702 THEN 'PF D'
                WHEN CD_PRFL = 2712 THEN 'PF E'
                WHEN CD_PRFL = 1111 THEN 'A CLASSIFICAR'
                ELSE 'A CLASSIFICAR'
            END AS NM_PRFL

        FROM vw_perfil_renda
    ),

    perfil_financeiro AS (
        SELECT
            CD_CLI,
            CD_MAC_PRFL_CLI,

            CASE
                WHEN CD_MAC_PRFL_CLI = 1 THEN 'Endividado'
                WHEN CD_MAC_PRFL_CLI = 2 THEN 'Equilibrista'
                WHEN CD_MAC_PRFL_CLI = 3 THEN 'Investidor'
                ELSE 'A CLASSIFICAR'
            END AS NM_MAC_PRFL_CLI,

            CD_MIC_PRFL_CLI,

            CASE
                WHEN CD_MIC_PRFL_CLI = 1 THEN 'Inadimplente'
                WHEN CD_MIC_PRFL_CLI = 2 THEN 'Acrobata'
                WHEN CD_MIC_PRFL_CLI = 3 THEN 'Iminente'
                WHEN CD_MIC_PRFL_CLI = 4 THEN 'Consciente'
                WHEN CD_MIC_PRFL_CLI = 5 THEN 'Equilibrista'
                WHEN CD_MIC_PRFL_CLI = 6 THEN 'Acelerado'
                WHEN CD_MIC_PRFL_CLI = 7 THEN 'Precavido'
                WHEN CD_MIC_PRFL_CLI = 8 THEN 'Despreocupado'
                ELSE 'A CLASSIFICAR'
            END AS NM_MIC_PRFL_CLI

        FROM vw_perfil_financeiro
    )

    SELECT
        b.*,

        pr.CD_PRFL,
        COALESCE(pr.NM_PRFL, 'A CLASSIFICAR') AS NM_PRFL,

        pf.CD_MAC_PRFL_CLI,
        COALESCE(
            pf.NM_MAC_PRFL_CLI,
            'A CLASSIFICAR'
        ) AS NM_MAC_PRFL_CLI,

        pf.CD_MIC_PRFL_CLI,
        COALESCE(
            pf.NM_MIC_PRFL_CLI,
            'A CLASSIFICAR'
        ) AS NM_MIC_PRFL_CLI,

        -- Convenção do código unificado: macro * 10 + micro.
        CASE
            WHEN COALESCE(
                pf.NM_MAC_PRFL_CLI,
                'A CLASSIFICAR'
            ) = 'A CLASSIFICAR'
            OR COALESCE(
                pf.NM_MIC_PRFL_CLI,
                'A CLASSIFICAR'
            ) = 'A CLASSIFICAR'
            THEN CAST(NULL AS BIGINT)

            ELSE CAST(
                (pf.CD_MAC_PRFL_CLI * 10)
                + pf.CD_MIC_PRFL_CLI
                AS BIGINT
            )
        END AS CD_PRFL_FIN,

        CASE
            WHEN COALESCE(
                pf.NM_MAC_PRFL_CLI,
                'A CLASSIFICAR'
            ) = 'A CLASSIFICAR'
            OR COALESCE(
                pf.NM_MIC_PRFL_CLI,
                'A CLASSIFICAR'
            ) = 'A CLASSIFICAR'
            THEN 'A CLASSIFICAR'

            ELSE CONCAT(
                pf.NM_MAC_PRFL_CLI,
                ' ',
                pf.NM_MIC_PRFL_CLI
            )
        END AS NM_PRFL_FIN

    FROM vw_resumo_transacoes_cliente b

    LEFT JOIN perfil_renda pr
        ON CAST(b.CD_CLI AS BIGINT) = pr.CD_CLI

    LEFT JOIN perfil_financeiro pf
        ON CAST(b.CD_CLI AS BIGINT) = pf.CD_CLI
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 8. Blocos classificados por cliente

Esta etapa é separada do resumo técnico. Ela sumariza as transações por `CD_CLASSIFICACAO_CATEGORIA` e produz uma linha por cliente para posterior incorporação na base oficial.

In [ ]:
%%spark

df_blocos_classificacao_cliente = spark.sql("""
    SELECT
        CD_CLI,

        -- Blocos de entrada
        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 1
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_REC,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 2
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_REEMB,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 3
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_RESG,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 0
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_IND,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 4
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_EMPR,

        -- Blocos de saída
        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 5
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_GEN,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 6
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_ESS,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 7
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_FLEX,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 8
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_RES,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 9
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_DIV

    FROM vw_base_transacoes

    GROUP BY
        CD_CLI
""")

df_blocos_classificacao_cliente.createOrReplaceTempView(
    "vw_blocos_classificacao_cliente"
)


### Incorporação dos blocos na sumarizada

Os totais `VL_ENT_TOTAL` e `VL_SAI_TOTAL` são calculados a partir dos blocos classificados. Essa é a última etapa que utiliza a base de transações antes dos indicadores derivados.

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    SELECT
        b.*,

        COALESCE(
            c.VL_ENT_REC,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_REC,

        COALESCE(
            c.VL_ENT_REEMB,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_REEMB,

        COALESCE(
            c.VL_ENT_RESG,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_RESG,

        COALESCE(
            c.VL_ENT_IND,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_IND,

        COALESCE(
            c.VL_ENT_EMPR,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_EMPR,

        CAST(
            COALESCE(c.VL_ENT_REC, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_ENT_REEMB, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_ENT_RESG, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_ENT_IND, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_ENT_EMPR, CAST(0 AS DECIMAL(18,2)))
            AS DECIMAL(18,2)
        ) AS VL_ENT_TOTAL,

        COALESCE(
            c.VL_SAI_GEN,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_GEN,

        COALESCE(
            c.VL_SAI_ESS,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_ESS,

        COALESCE(
            c.VL_SAI_FLEX,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_FLEX,

        COALESCE(
            c.VL_SAI_RES,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_RES,

        COALESCE(
            c.VL_SAI_DIV,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_DIV,

        CAST(
            COALESCE(c.VL_SAI_GEN, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_SAI_ESS, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_SAI_FLEX, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_SAI_RES, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_SAI_DIV, CAST(0 AS DECIMAL(18,2)))
            AS DECIMAL(18,2)
        ) AS VL_SAI_TOTAL

    FROM vw_base_sumarizada_cliente b

    LEFT JOIN vw_blocos_classificacao_cliente c
        ON CAST(b.CD_CLI AS BIGINT) = CAST(c.CD_CLI AS BIGINT)
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 9. Resultado do orçamento

O resultado é calculado exclusivamente a partir da sumarizada:

```text
VL_RES_ORC = VL_ENT_TOTAL - VL_SAI_TOTAL
PC_SAI_ENT = VL_SAI_TOTAL / VL_ENT_TOTAL
```

### Faixas de classificação

| Intervalo de `PC_SAI_ENT` | Resultado final |
|---:|---|
| `< 0,75` | Superavitário Forte |
| `0,75` até `< 0,95` | Superavitário Fraco |
| `0,95` até `< 1,00` | Neutro Fraco |
| `1,00` até `1,05` | Neutro Forte |
| `> 1,05` até `1,25` | Deficitário Fraco |
| `> 1,25` | Deficitário Forte |

Todas as divisões usam `NULLIF` e `COALESCE`: quando não houver base para a divisão, o percentual recebe `0`, sem erro.

Para os casos sem entrada:
- entrada e saída iguais a zero: `Neutro Fraco`;
- entrada igual a zero e saída positiva: `Deficitário Forte`.


In [ ]:
%%spark

df_resultado_orcamento = spark.sql(f"""
    WITH resultado_base AS (
        SELECT
            CD_CLI,

            CAST(
                COALESCE(VL_ENT_TOTAL, CAST(0 AS DECIMAL(18,2)))
                - COALESCE(VL_SAI_TOTAL, CAST(0 AS DECIMAL(18,2)))
                AS DECIMAL(18,2)
            ) AS VL_RES_ORC,

            CAST(
                COALESCE(
                    COALESCE(VL_SAI_TOTAL, CAST(0 AS DECIMAL(18,2)))
                    / NULLIF(
                        COALESCE(VL_ENT_TOTAL, CAST(0 AS DECIMAL(18,2))),
                        CAST(0 AS DECIMAL(18,2))
                    ),
                    CAST(0 AS DECIMAL(18,2))
                )
                AS DECIMAL(9,6)
            ) AS PC_SAI_ENT,

            COALESCE(
                VL_ENT_TOTAL,
                CAST(0 AS DECIMAL(18,2))
            ) AS VL_ENT_TOTAL,

            COALESCE(
                VL_SAI_TOTAL,
                CAST(0 AS DECIMAL(18,2))
            ) AS VL_SAI_TOTAL

        FROM vw_base_sumarizada_cliente
    ),

    resultado_classificado AS (
        SELECT
            CD_CLI,
            VL_RES_ORC,
            PC_SAI_ENT,

            CASE
                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL = 0
                THEN 0

                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL > 0
                THEN 2

                WHEN PC_SAI_ENT < {pc_limite_neutro_inferior}
                THEN 1

                WHEN PC_SAI_ENT > {pc_limite_neutro_superior}
                THEN 2

                ELSE 0
            END AS CD_RES_ORC,

            CASE
                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL = 0
                THEN 'Neutro'

                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL > 0
                THEN 'Deficitário'

                WHEN PC_SAI_ENT < {pc_limite_neutro_inferior}
                THEN 'Superavitário'

                WHEN PC_SAI_ENT > {pc_limite_neutro_superior}
                THEN 'Deficitário'

                ELSE 'Neutro'
            END AS TX_RES_ORC,

            CASE
                -- Sem entradas e sem saídas
                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL = 0
                THEN 'Fraco'

                -- Sem entrada e com saída
                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL > 0
                THEN 'Forte'

                -- Superavitário forte
                WHEN PC_SAI_ENT < {pc_limite_superavit_forte}
                THEN 'Forte'

                -- Superavitário fraco ou neutro fraco
                WHEN PC_SAI_ENT < 1
                THEN 'Fraco'

                -- Neutro forte
                WHEN PC_SAI_ENT <= {pc_limite_neutro_superior}
                THEN 'Forte'

                -- Deficitário fraco
                WHEN PC_SAI_ENT <= {pc_limite_deficit_forte}
                THEN 'Fraco'

                -- Deficitário forte
                ELSE 'Forte'
            END AS TX_STS_RES

        FROM resultado_base
    )

    SELECT
        CD_CLI,
        VL_RES_ORC,
        CAST(CD_RES_ORC AS INT) AS CD_RES_ORC,
        TX_RES_ORC,
        PC_SAI_ENT,
        TX_STS_RES,

        CONCAT(
            TX_RES_ORC,
            ' ',
            TX_STS_RES
        ) AS TX_STS_FINAL

    FROM resultado_classificado
""")

df_resultado_orcamento.createOrReplaceTempView(
    "vw_resultado_orcamento"
)


### Incorporação do resultado na sumarizada

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    SELECT
        b.*,

        r.VL_RES_ORC,
        r.CD_RES_ORC,
        r.TX_RES_ORC,
        r.PC_SAI_ENT,
        r.TX_STS_RES,
        r.TX_STS_FINAL

    FROM vw_base_sumarizada_cliente b

    INNER JOIN vw_resultado_orcamento r
        ON CAST(b.CD_CLI AS BIGINT) = CAST(r.CD_CLI AS BIGINT)
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 10. Percentuais de referência

Os percentuais de referência são fixos nesta primeira versão e são registrados na própria sumarizada para deixar explícita a régua utilizada por cliente.

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql(f"""
    SELECT
        b.*,

        CAST({pc_ref_gen} AS DECIMAL(9,6)) AS PC_REF_GEN,
        CAST({pc_ref_ess} AS DECIMAL(9,6)) AS PC_REF_ESS,
        CAST({pc_ref_flex} AS DECIMAL(9,6)) AS PC_REF_FLEX,
        CAST({pc_ref_res} AS DECIMAL(9,6)) AS PC_REF_RES,
        CAST({pc_ref_cred} AS DECIMAL(9,6)) AS PC_REF_CRED

    FROM vw_base_sumarizada_cliente b
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 11. Pontuação por concentração

Os percentuais por bloco são calculados apenas dentro desta consulta, sobre `VL_SAI_TOTAL`. Eles não são persistidos porque não fazem parte da estrutura final. Todas as divisões usam `NULLIF` e `COALESCE`. Sem saída no mês, os percentuais e as pontuações de concentração ficam em `0`.

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    WITH percentuais_saida AS (
        SELECT
            CD_CLI,

            PC_REF_GEN,
            PC_REF_ESS,
            PC_REF_FLEX,
            PC_REF_RES,
            PC_REF_CRED,

            COALESCE(
                VL_SAI_TOTAL,
                CAST(0 AS DECIMAL(18,2))
            ) AS VL_SAI_TOTAL,

            CAST(
                COALESCE(
                    COALESCE(VL_SAI_GEN, CAST(0 AS DECIMAL(18,2)))
                    / NULLIF(
                        COALESCE(VL_SAI_TOTAL, CAST(0 AS DECIMAL(18,2))),
                        CAST(0 AS DECIMAL(18,2))
                    ),
                    CAST(0 AS DECIMAL(18,2))
                )
                AS DECIMAL(9,6)
            ) AS PC_SAI_GEN,

            CAST(
                COALESCE(
                    COALESCE(VL_SAI_ESS, CAST(0 AS DECIMAL(18,2)))
                    / NULLIF(
                        COALESCE(VL_SAI_TOTAL, CAST(0 AS DECIMAL(18,2))),
                        CAST(0 AS DECIMAL(18,2))
                    ),
                    CAST(0 AS DECIMAL(18,2))
                )
                AS DECIMAL(9,6)
            ) AS PC_SAI_ESS,

            CAST(
                COALESCE(
                    COALESCE(VL_SAI_FLEX, CAST(0 AS DECIMAL(18,2)))
                    / NULLIF(
                        COALESCE(VL_SAI_TOTAL, CAST(0 AS DECIMAL(18,2))),
                        CAST(0 AS DECIMAL(18,2))
                    ),
                    CAST(0 AS DECIMAL(18,2))
                )
                AS DECIMAL(9,6)
            ) AS PC_SAI_FLEX,

            CAST(
                COALESCE(
                    COALESCE(VL_SAI_RES, CAST(0 AS DECIMAL(18,2)))
                    / NULLIF(
                        COALESCE(VL_SAI_TOTAL, CAST(0 AS DECIMAL(18,2))),
                        CAST(0 AS DECIMAL(18,2))
                    ),
                    CAST(0 AS DECIMAL(18,2))
                )
                AS DECIMAL(9,6)
            ) AS PC_SAI_RES,

            CAST(
                COALESCE(
                    COALESCE(VL_SAI_DIV, CAST(0 AS DECIMAL(18,2)))
                    / NULLIF(
                        COALESCE(VL_SAI_TOTAL, CAST(0 AS DECIMAL(18,2))),
                        CAST(0 AS DECIMAL(18,2))
                    ),
                    CAST(0 AS DECIMAL(18,2))
                )
                AS DECIMAL(9,6)
            ) AS PC_SAI_DIV

        FROM vw_base_sumarizada_cliente
    ),

    pontuacao_concentracao AS (
        SELECT
            CD_CLI,

            CAST(
                CASE
                    WHEN VL_SAI_TOTAL = 0 THEN 0
                    WHEN PC_SAI_GEN > PC_REF_GEN THEN 99
                    ELSE 0
                END
                AS INT
            ) AS NR_PONT_CONC_GEN,

            CAST(
                CASE
                    WHEN VL_SAI_TOTAL = 0 THEN 0
                    WHEN PC_SAI_ESS < PC_REF_ESS THEN 0
                    WHEN PC_SAI_ESS < (PC_REF_ESS * 1.5) THEN 1
                    ELSE 2
                END
                AS INT
            ) AS NR_PONT_CONC_ESS,

            CAST(
                CASE
                    WHEN VL_SAI_TOTAL = 0 THEN 0
                    WHEN PC_SAI_FLEX < PC_REF_FLEX THEN 0
                    WHEN PC_SAI_FLEX < (PC_REF_FLEX * 1.5) THEN 1
                    ELSE 2
                END
                AS INT
            ) AS NR_PONT_CONC_FLEX,

            CAST(
                CASE
                    WHEN VL_SAI_TOTAL = 0 THEN 0
                    WHEN PC_SAI_RES >= (PC_REF_RES * 1.5) THEN 0
                    WHEN PC_SAI_RES >= PC_REF_RES THEN 1
                    ELSE 2
                END
                AS INT
            ) AS NR_PONT_CONC_RES,

            CAST(
                CASE
                    WHEN VL_SAI_TOTAL = 0 THEN 0
                    WHEN PC_SAI_DIV < PC_REF_CRED THEN 0
                    WHEN PC_SAI_DIV < (PC_REF_CRED * 1.5) THEN 1
                    ELSE 2
                END
                AS INT
            ) AS NR_PONT_CONC_CRED

        FROM percentuais_saida
    )

    SELECT
        b.*,

        p.NR_PONT_CONC_GEN,
        p.NR_PONT_CONC_ESS,
        p.NR_PONT_CONC_FLEX,
        p.NR_PONT_CONC_RES,
        p.NR_PONT_CONC_CRED

    FROM vw_base_sumarizada_cliente b

    INNER JOIN pontuacao_concentracao p
        ON CAST(b.CD_CLI AS BIGINT) = CAST(p.CD_CLI AS BIGINT)
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 12. Pontua??o do or?amento

A classifica??o do or?amento continua usando `TX_RES_ORC` e `TX_STS_RES`, j? calculados na sumarizada. Na V2, a pontua??o de or?amento passa a usar apenas `TX_RES_ORC`: forte e fraco permanecem dispon?veis para an?lise, mas n?o alteram a pontua??o.

A pontua??o de or?amento pode acrescentar no m?ximo `1` ponto por tema. Resultado deficit?rio refor?a Gest?o do Or?amento, Consumo Planejado e Cr?dito. Resultado superavit?rio refor?a Forma??o de Reserva. Resultado neutro n?o acrescenta ponto.


In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    SELECT
        b.*,

        CAST(0 AS INT) AS NR_PONT_ORC_GEN,

        CAST(
            CASE
                WHEN TX_RES_ORC = 'Deficit?rio'
                THEN 1

                ELSE 0
            END
            AS INT
        ) AS NR_PONT_ORC_ESS,

        CAST(
            CASE
                WHEN TX_RES_ORC = 'Deficit?rio'
                THEN 1

                ELSE 0
            END
            AS INT
        ) AS NR_PONT_ORC_FLEX,

        CAST(
            CASE
                WHEN TX_RES_ORC = 'Superavit?rio'
                THEN 1

                ELSE 0
            END
            AS INT
        ) AS NR_PONT_ORC_RES,

        CAST(
            CASE
                WHEN TX_RES_ORC = 'Deficit?rio'
                THEN 1

                ELSE 0
            END
            AS INT
        ) AS NR_PONT_ORC_CRED

    FROM vw_base_sumarizada_cliente b
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 13. Pontua??o do perfil

A regra usa preferencialmente `NM_PRFL_FIN`, que combina macroperfil e subperfil. O perfil pode acrescentar no m?ximo `1` ponto e cada perfil completo refor?a no m?ximo um tema.

Quando o macroperfil estiver reconhecido, mas o subperfil estiver ausente, inv?lido ou `A CLASSIFICAR`, aplica-se fallback por macroperfil: `Endividado` refor?a Cr?dito, `Equilibrista` refor?a Reserva e `Investidor` n?o acrescenta ponto.


In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    SELECT
        b.*,

        CAST(0 AS INT) AS NR_PONT_PRFL_GEN,

        CAST(
            CASE
                WHEN NM_PRFL_FIN = 'Endividado Acrobata'
                THEN 1

                ELSE 0
            END
            AS INT
        ) AS NR_PONT_PRFL_ESS,

        CAST(
            CASE
                WHEN NM_PRFL_FIN = 'Endividado Consciente'
                THEN 1

                ELSE 0
            END
            AS INT
        ) AS NR_PONT_PRFL_FLEX,

        CAST(
            CASE
                WHEN NM_PRFL_FIN IN (
                    'Equilibrista',
                    'Equilibrista Equilibrista'
                )
                THEN 1

                WHEN NM_MAC_PRFL_CLI = 'Equilibrista'
                 AND COALESCE(NM_MIC_PRFL_CLI, 'A CLASSIFICAR') = 'A CLASSIFICAR'
                THEN 1

                ELSE 0
            END
            AS INT
        ) AS NR_PONT_PRFL_RES,

        CAST(
            CASE
                WHEN NM_PRFL_FIN IN (
                    'Endividado Iminente',
                    'Endividado Inadimplente'
                )
                THEN 1

                WHEN NM_MAC_PRFL_CLI = 'Endividado'
                 AND COALESCE(NM_MIC_PRFL_CLI, 'A CLASSIFICAR') = 'A CLASSIFICAR'
                THEN 1

                ELSE 0
            END
            AS INT
        ) AS NR_PONT_PRFL_CRED

    FROM vw_base_sumarizada_cliente b
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 14. Pontua??o final por tema

Na V2, a pontua??o por concentra??o ? a condi??o de entrada de cada tema. Para os quatro temas principais, quando a concentra??o do pr?prio tema for `0`, a pontua??o final do tema tamb?m ser? `0`, mesmo que or?amento ou perfil tenham pontuado.

Quando a concentra??o for maior que `0`, a pontua??o final soma concentra??o, or?amento e perfil. A categoriza??o permanece igual ? concentra??o gen?rica. Nenhuma regra de desempate ? aplicada nesta etapa; empates devem permanecer dispon?veis para avalia??o posterior.


In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    SELECT
        b.*,

        CAST(
            COALESCE(NR_PONT_CONC_GEN, 0)
            AS INT
        ) AS NR_PONT_CATEG,

        CAST(
            CASE
                WHEN COALESCE(NR_PONT_CONC_ESS, 0) = 0
                THEN 0

                ELSE COALESCE(NR_PONT_CONC_ESS, 0)
                    + COALESCE(NR_PONT_ORC_ESS, 0)
                    + COALESCE(NR_PONT_PRFL_ESS, 0)
            END
            AS INT
        ) AS NR_PONT_ORC,

        CAST(
            CASE
                WHEN COALESCE(NR_PONT_CONC_FLEX, 0) = 0
                THEN 0

                ELSE COALESCE(NR_PONT_CONC_FLEX, 0)
                    + COALESCE(NR_PONT_ORC_FLEX, 0)
                    + COALESCE(NR_PONT_PRFL_FLEX, 0)
            END
            AS INT
        ) AS NR_PONT_CONS,

        CAST(
            CASE
                WHEN COALESCE(NR_PONT_CONC_RES, 0) = 0
                THEN 0

                ELSE COALESCE(NR_PONT_CONC_RES, 0)
                    + COALESCE(NR_PONT_ORC_RES, 0)
                    + COALESCE(NR_PONT_PRFL_RES, 0)
            END
            AS INT
        ) AS NR_PONT_RES,

        CAST(
            CASE
                WHEN COALESCE(NR_PONT_CONC_CRED, 0) = 0
                THEN 0

                ELSE COALESCE(NR_PONT_CONC_CRED, 0)
                    + COALESCE(NR_PONT_ORC_CRED, 0)
                    + COALESCE(NR_PONT_PRFL_CRED, 0)
            END
            AS INT
        ) AS NR_PONT_CRED

    FROM vw_base_sumarizada_cliente b
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 15. Marcação de movimentação agro e participação no radar

As regras ficam concentradas em um único bloco, com etapas separadas internamente:

- `marcacao_agro_cliente`: identifica movimentação `C` ou `D` em categoria marcada como agro;
- `base_com_marcacao_agro`: incorpora a marca por cliente na sumarizada;
- `regras_participacao_radar`: aplica os critérios mínimos de participação.

Cliente com movimentação agro não participa do radar.


In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    WITH marcacao_agro_cliente AS (
        SELECT
            CD_CLI,

            CASE
                WHEN MAX(
                    CASE
                        WHEN FL_AGRO_CATEGORIA = 'S'
                         AND CD_NTZ_CTB_TRAN IN ('C', 'D')
                        THEN 1
                        ELSE 0
                    END
                ) = 1
                THEN 'S'

                ELSE 'N'
            END AS FL_TEM_MOV_AGRO

        FROM vw_base_transacoes

        GROUP BY
            CD_CLI
    ),

    base_com_marcacao_agro AS (
        SELECT
            b.*,

            COALESCE(
                a.FL_TEM_MOV_AGRO,
                'N'
            ) AS FL_TEM_MOV_AGRO

        FROM vw_base_sumarizada_cliente b

        LEFT JOIN marcacao_agro_cliente a
            ON CAST(b.CD_CLI AS BIGINT) = CAST(a.CD_CLI AS BIGINT)
    ),

    regras_participacao_radar AS (
        SELECT
            b.*,

            CASE
                WHEN QT_TRANS_TOTAL > 0
                 AND QT_TRANS_ENT > 0
                 AND QT_TRANS_SAI > 0
                 AND VL_TRANS_ENT > 0
                 AND VL_TRANS_SAI > 0
                 AND COALESCE(NM_PRFL, 'A CLASSIFICAR')
                     <> 'A CLASSIFICAR'
                 AND COALESCE(NM_MAC_PRFL_CLI, 'A CLASSIFICAR')
                     <> 'A CLASSIFICAR'
                 AND COALESCE(NM_MIC_PRFL_CLI, 'A CLASSIFICAR')
                     <> 'A CLASSIFICAR'
                 AND COALESCE(NM_PRFL_FIN, 'A CLASSIFICAR')
                     <> 'A CLASSIFICAR'
                 AND FL_TEM_MOV_AGRO = 'N'
                THEN 'S'

                ELSE 'N'
            END AS FL_PARTICIPA_RADAR

        FROM base_com_marcacao_agro b
    )

    SELECT
        *

    FROM regras_participacao_radar
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 16. Ordem física final da sumarizada

A seleção explícita garante que a ordem do DataFrame seja a mesma do DDL e da carga final na tabela física.


In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    SELECT
        CAST(CD_CLI AS INT) AS CD_CLI,

        QT_TRANS_TOTAL,

        QT_TRANS_ENT,
        QTD_TRANS_ENT_CC,
        QTD_TRANS_ENT_CD,

        QT_TRANS_SAI,
        QTD_TRANS_SAI_CC,
        QTD_TRANS_SAI_CD,

        VL_TRANS_ENT,
        VL_TRANS_ENT_CC,
        VL_TRANS_ENT_CD,

        VL_TRANS_SAI,
        VL_TRANS_SAI_CC,
        VL_TRANS_SAI_CD,

        CD_PRFL,
        NM_PRFL,

        CD_MAC_PRFL_CLI,
        NM_MAC_PRFL_CLI,

        CD_MIC_PRFL_CLI,
        NM_MIC_PRFL_CLI,

        CD_PRFL_FIN,
        NM_PRFL_FIN,

        VL_ENT_REC,
        VL_ENT_REEMB,
        VL_ENT_RESG,
        VL_ENT_IND,
        VL_ENT_EMPR,
        VL_ENT_TOTAL,

        VL_SAI_GEN,
        VL_SAI_ESS,
        VL_SAI_FLEX,
        VL_SAI_RES,
        VL_SAI_DIV,
        VL_SAI_TOTAL,

        VL_RES_ORC,
        CD_RES_ORC,
        TX_RES_ORC,
        PC_SAI_ENT,
        TX_STS_RES,
        TX_STS_FINAL,

        PC_REF_GEN,
        PC_REF_ESS,
        PC_REF_FLEX,
        PC_REF_RES,
        PC_REF_CRED,

        NR_PONT_CONC_GEN,
        NR_PONT_CONC_ESS,
        NR_PONT_CONC_FLEX,
        NR_PONT_CONC_RES,
        NR_PONT_CONC_CRED,

        NR_PONT_ORC_GEN,
        NR_PONT_ORC_ESS,
        NR_PONT_ORC_FLEX,
        NR_PONT_ORC_RES,
        NR_PONT_ORC_CRED,

        NR_PONT_PRFL_GEN,
        NR_PONT_PRFL_ESS,
        NR_PONT_PRFL_FLEX,
        NR_PONT_PRFL_RES,
        NR_PONT_PRFL_CRED,

        NR_PONT_CATEG,
        NR_PONT_ORC,
        NR_PONT_CONS,
        NR_PONT_RES,
        NR_PONT_CRED,

        FL_TEM_MOV_AGRO,
        FL_PARTICIPA_RADAR

    FROM vw_base_sumarizada_cliente
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 17. Datas de execução e referência

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql(f"""
    SELECT
        b.*,

        CAST('{data_atual}' AS DATE) AS DT_EXEA,

        TRUNC(
            CAST('{data_atual}' AS DATE),
            'MONTH'
        ) AS DT_MES_EXEA,

        CAST('{data_ini}' AS DATE) AS DT_REF_INI,

        CAST('{data_fim}' AS DATE) AS DT_REF_FIM

    FROM vw_base_sumarizada_cliente b
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)

## 18. Validações finais mínimas

As validações são executadas somente ao final do fluxo e confirmam: uma linha por cliente, manutenção da quantidade de clientes, coerência entre totais classificados e resultado do orçamento, além da ordem física esperada para a carga.


In [ ]:
%%spark

colunas_esperadas = ['CD_CLI', 'QT_TRANS_TOTAL', 'QT_TRANS_ENT', 'QTD_TRANS_ENT_CC', 'QTD_TRANS_ENT_CD', 'QT_TRANS_SAI', 'QTD_TRANS_SAI_CC', 'QTD_TRANS_SAI_CD', 'VL_TRANS_ENT', 'VL_TRANS_ENT_CC', 'VL_TRANS_ENT_CD', 'VL_TRANS_SAI', 'VL_TRANS_SAI_CC', 'VL_TRANS_SAI_CD', 'CD_PRFL', 'NM_PRFL', 'CD_MAC_PRFL_CLI', 'NM_MAC_PRFL_CLI', 'CD_MIC_PRFL_CLI', 'NM_MIC_PRFL_CLI', 'CD_PRFL_FIN', 'NM_PRFL_FIN', 'VL_ENT_REC', 'VL_ENT_REEMB', 'VL_ENT_RESG', 'VL_ENT_IND', 'VL_ENT_EMPR', 'VL_ENT_TOTAL', 'VL_SAI_GEN', 'VL_SAI_ESS', 'VL_SAI_FLEX', 'VL_SAI_RES', 'VL_SAI_DIV', 'VL_SAI_TOTAL', 'VL_RES_ORC', 'CD_RES_ORC', 'TX_RES_ORC', 'PC_SAI_ENT', 'TX_STS_RES', 'TX_STS_FINAL', 'PC_REF_GEN', 'PC_REF_ESS', 'PC_REF_FLEX', 'PC_REF_RES', 'PC_REF_CRED', 'NR_PONT_CONC_GEN', 'NR_PONT_CONC_ESS', 'NR_PONT_CONC_FLEX', 'NR_PONT_CONC_RES', 'NR_PONT_CONC_CRED', 'NR_PONT_ORC_GEN', 'NR_PONT_ORC_ESS', 'NR_PONT_ORC_FLEX', 'NR_PONT_ORC_RES', 'NR_PONT_ORC_CRED', 'NR_PONT_PRFL_GEN', 'NR_PONT_PRFL_ESS', 'NR_PONT_PRFL_FLEX', 'NR_PONT_PRFL_RES', 'NR_PONT_PRFL_CRED', 'NR_PONT_CATEG', 'NR_PONT_ORC', 'NR_PONT_CONS', 'NR_PONT_RES', 'NR_PONT_CRED', 'FL_TEM_MOV_AGRO', 'FL_PARTICIPA_RADAR','DT_EXEA','DT_MES_EXEA','DT_REF_INI','DT_REF_FIM']

if df_base_sumarizada_cliente.columns != colunas_esperadas:
    raise ValueError(
        "A ordem ou os nomes das colunas da base sumarizada diferem "
        "da estrutura final esperada."
    )

qt_resumo_transacoes = df_resumo_transacoes_cliente.count()

linha_validacao = spark.sql("""
    SELECT
        COUNT(*) AS QT_LINHAS,
        COUNT(DISTINCT CD_CLI) AS QT_CLIENTES_DISTINTOS,

        SUM(
            CASE
                WHEN VL_TRANS_ENT <> CAST(VL_ENT_TOTAL AS DECIMAL(25,2))
                  OR VL_TRANS_SAI <> CAST(VL_SAI_TOTAL AS DECIMAL(25,2))
                THEN 1
                ELSE 0
            END
        ) AS QT_INCONSISTENCIA_BLOCOS,

        SUM(
            CASE
                WHEN VL_RES_ORC <> CAST(
                    VL_ENT_TOTAL - VL_SAI_TOTAL
                    AS DECIMAL(18,2)
                )
                OR PC_SAI_ENT <> CAST(
                    COALESCE(
                        COALESCE(
                            VL_SAI_TOTAL,
                            CAST(0 AS DECIMAL(18,2))
                        )
                        / NULLIF(
                            COALESCE(
                                VL_ENT_TOTAL,
                                CAST(0 AS DECIMAL(18,2))
                            ),
                            CAST(0 AS DECIMAL(18,2))
                        ),
                        CAST(0 AS DECIMAL(18,2))
                    )
                    AS DECIMAL(9,6)
                )
                THEN 1
                ELSE 0
            END
        ) AS QT_INCONSISTENCIA_RESULTADO

    FROM vw_base_sumarizada_cliente
""").first()

qt_base_sumarizada = linha_validacao["QT_LINHAS"]
qt_clientes_distintos = linha_validacao["QT_CLIENTES_DISTINTOS"]
qt_inconsistencia_blocos = linha_validacao["QT_INCONSISTENCIA_BLOCOS"] or 0
qt_inconsistencia_resultado = (
    linha_validacao["QT_INCONSISTENCIA_RESULTADO"] or 0
)

if qt_resumo_transacoes != qt_base_sumarizada:
    raise ValueError(
        "A quantidade de clientes foi alterada no fluxo: "
        f"resumo_transacoes={qt_resumo_transacoes}, "
        f"base_sumarizada={qt_base_sumarizada}."
    )

if qt_base_sumarizada != qt_clientes_distintos:
    raise ValueError(
        "A base sumarizada possui mais de uma linha para o mesmo CD_CLI."
    )

if qt_inconsistencia_blocos > 0:
    raise ValueError(
        "Os totais de entrada ou saída diferem da soma dos blocos classificados."
    )

if qt_inconsistencia_resultado > 0:
    raise ValueError(
        "O resultado do orçamento ou PC_SAI_ENT não está coerente."
    )

print(
    "Base sumarizada validada com sucesso: "
    f"{qt_base_sumarizada} clientes, uma linha por CD_CLI, "
    f"{len(colunas_esperadas)} colunas."
)

df_base_sumarizada_cliente.printSchema()


## 19. DDL final da tabela

A estrutura abaixo reflete exatamente a ordem da `df_base_sumarizada_cliente` validada na etapa anterior.


In [ ]:
%%spark

ddl_tabela_spark = f"""
CREATE TABLE {tabela_spark} (

    -- Identificação
    CD_CLI                     INT             COMMENT 'Código do cliente',

    -- Quantidades de transações
    QT_TRANS_TOTAL             BIGINT          COMMENT 'Quantidade total de transações',

    QT_TRANS_ENT               BIGINT          COMMENT 'Quantidade de transações de entrada',
    QTD_TRANS_ENT_CC           BIGINT          COMMENT 'Quantidade de transações de entrada em conta corrente',
    QTD_TRANS_ENT_CD           BIGINT          COMMENT 'Quantidade de transações de entrada em cartão',

    QT_TRANS_SAI               BIGINT          COMMENT 'Quantidade de transações de saída',
    QTD_TRANS_SAI_CC           BIGINT          COMMENT 'Quantidade de transações de saída em conta corrente',
    QTD_TRANS_SAI_CD           BIGINT          COMMENT 'Quantidade de transações de saída em cartão',

    -- Valores de transações
    VL_TRANS_ENT               DECIMAL(25,2)   COMMENT 'Valor total de entradas',
    VL_TRANS_ENT_CC            DECIMAL(25,2)   COMMENT 'Valor de entradas em conta corrente',
    VL_TRANS_ENT_CD            DECIMAL(25,2)   COMMENT 'Valor de entradas em cartão',

    VL_TRANS_SAI               DECIMAL(25,2)   COMMENT 'Valor total de saídas',
    VL_TRANS_SAI_CC            DECIMAL(25,2)   COMMENT 'Valor de saídas em conta corrente',
    VL_TRANS_SAI_CD            DECIMAL(25,2)   COMMENT 'Valor de saídas em cartão',

    -- Perfil de renda
    CD_PRFL                    INT             COMMENT 'Código do perfil de renda',
    NM_PRFL                    STRING          COMMENT 'Texto do perfil de renda',

    -- Perfil financeiro
    CD_MAC_PRFL_CLI            BIGINT          COMMENT 'Código do macroperfil financeiro',
    NM_MAC_PRFL_CLI            STRING          COMMENT 'Texto do macroperfil financeiro',

    CD_MIC_PRFL_CLI            BIGINT          COMMENT 'Código do microperfil financeiro',
    NM_MIC_PRFL_CLI            STRING          COMMENT 'Texto do microperfil financeiro',

    -- Perfil financeiro unificado
    CD_PRFL_FIN                BIGINT          COMMENT 'Código derivado do perfil financeiro: macro x 10 + micro',
    NM_PRFL_FIN                STRING          COMMENT 'Texto unificado do perfil financeiro',

    -- Blocos de entrada
    VL_ENT_REC                 DECIMAL(18,2)   COMMENT 'Valor de entrada por receita, rendimento ou benefício',
    VL_ENT_REEMB               DECIMAL(18,2)   COMMENT 'Valor de entrada por restituição, estorno ou ajuste',
    VL_ENT_RESG                DECIMAL(18,2)   COMMENT 'Valor de entrada por resgate de investimento',
    VL_ENT_IND                 DECIMAL(18,2)   COMMENT 'Valor de entrada por transferência ou entrada indefinida',
    VL_ENT_EMPR                DECIMAL(18,2)   COMMENT 'Valor de entrada por empréstimo ou crédito liberado',
    VL_ENT_TOTAL               DECIMAL(18,2)   COMMENT 'Valor total de entrada',

    -- Blocos de saída
    VL_SAI_GEN                 DECIMAL(18,2)   COMMENT 'Valor de saída genérica ou não classificada',
    VL_SAI_ESS                 DECIMAL(18,2)   COMMENT 'Valor de saída essencial',
    VL_SAI_FLEX                DECIMAL(18,2)   COMMENT 'Valor de saída flexível',
    VL_SAI_RES                 DECIMAL(18,2)   COMMENT 'Valor de saída para reserva ou futuro',
    VL_SAI_DIV                 DECIMAL(18,2)   COMMENT 'Valor de saída para dívidas, crédito ou custo financeiro',
    VL_SAI_TOTAL               DECIMAL(18,2)   COMMENT 'Valor total de saída',

    -- Resultado do orçamento
    VL_RES_ORC                 DECIMAL(18,2)   COMMENT 'Valor do resultado do orçamento: entradas menos saídas',
    CD_RES_ORC                 INT             COMMENT 'Código do resultado do orçamento: 0 = Neutro; 1 = Superavitário; 2 = Deficitário',
    TX_RES_ORC                 STRING          COMMENT 'Texto do resultado do orçamento',
    PC_SAI_ENT                 DECIMAL(9,6)    COMMENT 'Percentual de saída sobre entrada',
    TX_STS_RES                 STRING          COMMENT 'Status da intensidade do resultado: Forte ou Fraco',
    TX_STS_FINAL               STRING          COMMENT 'Texto final composto pelo resultado e seu status',

    -- Percentuais de referência
    PC_REF_GEN                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída genérica',
    PC_REF_ESS                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída essencial',
    PC_REF_FLEX                DECIMAL(9,6)    COMMENT 'Percentual de referência para saída flexível',
    PC_REF_RES                 DECIMAL(9,6)    COMMENT 'Percentual de referência para reserva',
    PC_REF_CRED                DECIMAL(9,6)    COMMENT 'Percentual de referência para crédito',

    -- Pontuação por concentração
    NR_PONT_CONC_GEN           INT             COMMENT 'Pontuação de concentração genérica',
    NR_PONT_CONC_ESS           INT             COMMENT 'Pontuação de concentração essencial',
    NR_PONT_CONC_FLEX          INT             COMMENT 'Pontuação de concentração flexível',
    NR_PONT_CONC_RES           INT             COMMENT 'Pontuação de concentração reserva',
    NR_PONT_CONC_CRED          INT             COMMENT 'Pontuação de concentração crédito',

    -- Pontuação do orçamento
    NR_PONT_ORC_GEN            INT             COMMENT 'Pontuação de orçamento genérica',
    NR_PONT_ORC_ESS            INT             COMMENT 'Pontuação de orçamento essencial',
    NR_PONT_ORC_FLEX           INT             COMMENT 'Pontuação de orçamento flexível',
    NR_PONT_ORC_RES            INT             COMMENT 'Pontuação de orçamento reserva',
    NR_PONT_ORC_CRED           INT             COMMENT 'Pontuação de orçamento crédito',

    -- Pontuação do perfil
    NR_PONT_PRFL_GEN           INT             COMMENT 'Pontuação de perfil genérica',
    NR_PONT_PRFL_ESS           INT             COMMENT 'Pontuação de perfil essencial',
    NR_PONT_PRFL_FLEX          INT             COMMENT 'Pontuação de perfil flexível',
    NR_PONT_PRFL_RES           INT             COMMENT 'Pontuação de perfil reserva',
    NR_PONT_PRFL_CRED          INT             COMMENT 'Pontuação de perfil crédito',

    -- Pontuação final por tema
    NR_PONT_CATEG              INT             COMMENT 'Pontuação final de categorização de gastos',
    NR_PONT_ORC                INT             COMMENT 'Pontuação final de gestão de orçamento',
    NR_PONT_CONS               INT             COMMENT 'Pontuação final de consumo planejado',
    NR_PONT_RES                INT             COMMENT 'Pontuação final de formação de reserva',
    NR_PONT_CRED               INT             COMMENT 'Pontuação final de uso consciente do crédito',

    -- Marcação de contexto e participação no radar
    FL_TEM_MOV_AGRO            STRING          COMMENT 'Indica se o cliente teve movimentação de crédito ou débito em categoria marcada como agro: S ou N',
    FL_PARTICIPA_RADAR         STRING          COMMENT 'Indica se o cliente atende às regras mínimas para participar do radar: S ou N',

    -- Datas de execução e referência

    DT_EXEA                    DATE            COMMENT 'Data de execução do ETL',

    DT_MES_EXEA                DATE            COMMENT 'Mês de execução do ETL, representado pelo primeiro dia do mês',

    DT_REF_INI                 DATE            COMMENT 'Data inicial do período de referência analisado',

    DT_REF_FIM                 DATE            COMMENT 'Data final do período de referência analisado'
)
COMMENT 'Análise de Educação Financeira do Cliente'
STORED AS PARQUET
TBLPROPERTIES (
    'parquet.compress' = 'SNAPPY'
)
"""

print(ddl_tabela_spark)


## 20. Carga final da tabela

A carga é integral e usa `overwrite`. Com `atualizar_metadado = True`, a tabela é recriada com o DDL desta versão antes da carga.


In [ ]:
%%spark

atualizar_metadado = True

if atualizar_metadado:
    spark.sql(f"DROP TABLE IF EXISTS {tabela_spark}")
    spark.sql(ddl_tabela_spark)

colunas_carga = [
        "CD_CLI",
        "QT_TRANS_TOTAL",
        "QT_TRANS_ENT",
        "QTD_TRANS_ENT_CC",
        "QTD_TRANS_ENT_CD",
        "QT_TRANS_SAI",
        "QTD_TRANS_SAI_CC",
        "QTD_TRANS_SAI_CD",
        "VL_TRANS_ENT",
        "VL_TRANS_ENT_CC",
        "VL_TRANS_ENT_CD",
        "VL_TRANS_SAI",
        "VL_TRANS_SAI_CC",
        "VL_TRANS_SAI_CD",
        "CD_PRFL",
        "NM_PRFL",
        "CD_MAC_PRFL_CLI",
        "NM_MAC_PRFL_CLI",
        "CD_MIC_PRFL_CLI",
        "NM_MIC_PRFL_CLI",
        "CD_PRFL_FIN",
        "NM_PRFL_FIN",
        "VL_ENT_REC",
        "VL_ENT_REEMB",
        "VL_ENT_RESG",
        "VL_ENT_IND",
        "VL_ENT_EMPR",
        "VL_ENT_TOTAL",
        "VL_SAI_GEN",
        "VL_SAI_ESS",
        "VL_SAI_FLEX",
        "VL_SAI_RES",
        "VL_SAI_DIV",
        "VL_SAI_TOTAL",
        "VL_RES_ORC",
        "CD_RES_ORC",
        "TX_RES_ORC",
        "PC_SAI_ENT",
        "TX_STS_RES",
        "TX_STS_FINAL",
        "PC_REF_GEN",
        "PC_REF_ESS",
        "PC_REF_FLEX",
        "PC_REF_RES",
        "PC_REF_CRED",
        "NR_PONT_CONC_GEN",
        "NR_PONT_CONC_ESS",
        "NR_PONT_CONC_FLEX",
        "NR_PONT_CONC_RES",
        "NR_PONT_CONC_CRED",
        "NR_PONT_ORC_GEN",
        "NR_PONT_ORC_ESS",
        "NR_PONT_ORC_FLEX",
        "NR_PONT_ORC_RES",
        "NR_PONT_ORC_CRED",
        "NR_PONT_PRFL_GEN",
        "NR_PONT_PRFL_ESS",
        "NR_PONT_PRFL_FLEX",
        "NR_PONT_PRFL_RES",
        "NR_PONT_PRFL_CRED",
        "NR_PONT_CATEG",
        "NR_PONT_ORC",
        "NR_PONT_CONS",
        "NR_PONT_RES",
        "NR_PONT_CRED",
        "FL_TEM_MOV_AGRO",
        "FL_PARTICIPA_RADAR",
        "DT_EXEA",
        "DT_MES_EXEA",
        "DT_REF_INI",
        "DT_REF_FIM",
]

(
    df_base_sumarizada_cliente
    .select(*colunas_carga)
    .write
    .mode("overwrite")
    .insertInto(tabela_spark)
)

print(
    "Tabela carregada com sucesso: "
    f"{tabela_spark}"
)
